In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

In [2]:
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [3]:
import kagglehub

dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(dataset_dir)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
/kaggle/input/brazilian-ecommerce


In [4]:
import os

files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(
        table_name,
        engine,
        if_exists='replace',
        index=False,
        chunksize=10_000
    )
    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


## Завдання 1. Setup і typed-схема Olist

Датасет Olist було завантажено з Kaggle. П'ять CSV-файлів завантажено у raw-таблиці PostgreSQL перед створенням typed-схеми.

In [5]:
typed_schema_sql = """
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders CASCADE;
DROP TABLE IF EXISTS olist_customers CASCADE;
DROP TABLE IF EXISTS olist_products CASCADE;
DROP TABLE IF EXISTS olist_sellers CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);
"""

with engine.begin() as conn:
    conn.execute(text(typed_schema_sql))

print("Typed-схему створено успішно.")

Typed-схему створено успішно.


In [6]:
insert_typed_sql = """
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;
"""

with engine.begin() as conn:
    conn.execute(text(insert_typed_sql))

print("Дані перенесено у typed-таблиці.")

Дані перенесено у typed-таблиці.


In [7]:
row_counts_sql = """
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL
SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL
SELECT 'products', COUNT(*) FROM olist_products
UNION ALL
SELECT 'sellers', COUNT(*) FROM olist_sellers
ORDER BY table_name;
"""

pd.read_sql(row_counts_sql, engine)

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,sellers,3095


## Завдання 2. CTE-chain pipeline

На цьому етапі я створюю customer-level ознаки за допомогою послідовного CTE-chain. Кожен етап виконує окрему частину трансформації даних.

In [8]:
cte_query = """
WITH
reference_date AS (
    -- Stage 0: стабільна reference date для історичного датасету
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),

stage1_orders_total AS (
    -- Stage 1: рахуємо загальну суму кожного замовлення
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),

stage2_rfm AS (
    -- Stage 2: рахуємо RFM для кожного клієнта
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY
        s.customer_id,
        rd.as_of_date
),

stage3_cohort AS (
    -- Stage 3: визначаємо квартал першого замовлення
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),

stage4_rolling_aov AS (
    -- Stage 4: середня сума замовлення за останні 30 днів
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),

stage5_cancel_rate AS (
    -- Stage 5: частка скасованих замовлень для кожного клієнта
    SELECT
        customer_id,
        AVG(
            CASE
                WHEN order_status = 'canceled' THEN 1.0
                ELSE 0.0
            END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),

stage6_category_revenue AS (
    -- Stage 6: рахуємо дохід по категоріях для кожного клієнта
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)')
),

stage7_preferred_category AS (
    -- Stage 7: визначаємо улюблену категорію клієнта
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)

SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2)
        AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4)
        AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)')
        AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2)
        AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 20;
"""

pd.read_sql(cte_query, engine)

,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66
5,05455dfa7cd02f13d132aa7a6a9729c6,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0,agro_industria_e_comercio,6081.54
6,df55c14d1476a9a3467f131269c2477f,RJ,565,1,4950.34,2017-04-01,2017-04-01,4950.34,0.0,eletroportateis,4950.34
7,e0a2412720e9ea4f26c1ac985f6a7358,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0,cool_stuff,4809.44
8,24bbf5fd2f2e1b359ee7de94defc4a15,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0,eletroportateis,4764.34
9,3d979689f636322c62418b6346b1c6d2,PB,118,1,4681.78,2018-06-22,2018-04-01,4681.78,0.0,pcs,4681.78


## Завдання 3. Рекурсивний CTE

Оскільки в датасеті Olist немає даних про реферальні зв'язки між клієнтами, для демонстрації рекурсивного CTE створюю невелику синтетичну таблицю `hw5_referrals`.

In [9]:
referrals_sql = """
DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH ordered_customers AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT
    prev.customer_id AS referrer_customer_id,
    nxt.customer_id AS referred_customer_id
FROM ordered_customers AS prev
JOIN ordered_customers AS nxt
    ON nxt.rn = prev.rn + 1
ON CONFLICT (referred_customer_id) DO NOTHING;
"""

with engine.begin() as conn:
    conn.execute(text(referrals_sql))

print("Таблицю hw5_referrals створено.")

Таблицю hw5_referrals створено.


In [10]:
check_referrals_sql = """
SELECT *
FROM hw5_referrals
ORDER BY referrer_customer_id, referred_customer_id;
"""

pd.read_sql(check_referrals_sql, engine)

,referrer_customer_id,referred_customer_id,referred_at
0,00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140,2026-10-05 12:08:54.907843+00:00
1,000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079,2026-10-05 12:08:54.907843+00:00
2,0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5,2026-10-05 12:08:54.907843+00:00
3,0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb,2026-10-05 12:08:54.907843+00:00
4,000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652,2026-10-05 12:08:54.907843+00:00
5,0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086,2026-10-05 12:08:54.907843+00:00
6,000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282,2026-10-05 12:08:54.907843+00:00
7,00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c,2026-10-05 12:08:54.907843+00:00
8,00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130,2026-10-05 12:08:54.907843+00:00


In [11]:
recursive_query = """
WITH RECURSIVE referral_depth AS (
    -- ANCHOR: клієнти, яких не запросили інші клієнти
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: переходимо від referrer до referred customer
    SELECT
        r.referred_customer_id AS customer_id,
        rd.depth + 1 AS depth,
        rd.path_ids || r.referred_customer_id AS path_ids,
        rd.path_text || ' -> ' || r.referred_customer_id AS path_text
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON rd.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(rd.path_ids)
      AND rd.depth < 50
)

SELECT
    depth,
    COUNT(*) AS n_customers,
    MAX(path_text) AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;
"""

pd.read_sql(recursive_query, engine)

,depth,n_customers,sample_path
0,1,99432,ffffe8b65bbe3087b653a978c870db99
1,2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
2,3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
3,4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
4,5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
5,6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
6,7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
7,8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
8,9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
9,10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...


## Завдання 4. Materialized View

На цьому етапі створюю Materialized View на основі повного CTE-pipeline з попереднього завдання. Це дозволяє зберігати вже розраховані customer-level ознаки та швидше отримувати їх у наступних запитах.

In [12]:
mv_sql = """
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

CREATE MATERIALIZED VIEW mv_olist_customer_features AS
WITH
reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),

stage1_orders_total AS (
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),

stage2_rfm AS (
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),

stage3_cohort AS (
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),

stage4_rolling_aov AS (
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),

stage5_cancel_rate AS (
    SELECT
        customer_id,
        AVG(
            CASE
                WHEN order_status = 'canceled' THEN 1.0
                ELSE 0.0
            END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),

stage6_category_revenue AS (
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)')
),

stage7_preferred_category AS (
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)

SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2)
        AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4)
        AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)')
        AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2)
        AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id;

CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features (customer_id);
"""

with engine.begin() as conn:
    conn.execute(text(mv_sql))

print("Materialized View та UNIQUE INDEX створено.")

Materialized View та UNIQUE INDEX створено.


In [13]:
check_mv_sql = """
SELECT COUNT(*) AS n_rows
FROM mv_olist_customer_features;
"""

pd.read_sql(check_mv_sql, engine)

,n_rows
0,99441


In [14]:
top_mv_sql = """
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;
"""

pd.read_sql(top_mv_sql, engine)

,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66


In [15]:
refresh_sql = """
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;
"""

with engine.begin() as conn:
    conn.execute(text(refresh_sql))

print("Materialized View успішно оновлено через CONCURRENTLY.")

Materialized View успішно оновлено через CONCURRENTLY.


## Завдання 5. EXPLAIN ANALYZE та Reflection

Порівнюю план виконання повного CTE-pipeline з читанням уже підготовлених даних із Materialized View за допомогою `EXPLAIN (ANALYZE, BUFFERS)`.

In [16]:
explain_cte_sql = """
EXPLAIN (ANALYZE, BUFFERS)
WITH
reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),

stage1_orders_total AS (
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),

stage2_rfm AS (
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY
        s.customer_id,
        rd.as_of_date
)

SELECT *
FROM stage2_rfm
ORDER BY monetary DESC NULLS LAST
LIMIT 10;
"""

with engine.connect() as conn:
    result = conn.execute(text(explain_cte_sql))
    explain_cte = [row[0] for row in result]

print("\n".join(explain_cte))

Limit  (cost=33552.08..33552.10 rows=10 width=64) (actual time=766.809..769.266 rows=10 loops=1)
  Buffers: shared hit=5966, temp read=3332 written=4071
  ->  Sort  (cost=33552.08..33552.58 rows=200 width=64) (actual time=766.807..769.262 rows=10 loops=1)
        Sort Key: stage2_rfm.monetary DESC NULLS LAST
        Sort Method: top-N heapsort  Memory: 27kB
        Buffers: shared hit=5966, temp read=3332 written=4071
        ->  Subquery Scan on stage2_rfm  (cost=33542.25..33547.75 rows=200 width=64) (actual time=608.925..742.952 rows=99441 loops=1)
              Buffers: shared hit=5966, temp read=3332 written=4071
              ->  HashAggregate  (cost=33542.25..33545.75 rows=200 width=68) (actual time=608.922..732.021 rows=99441 loops=1)
                    Group Key: o.customer_id, (((max(olist_orders.order_purchase_timestamp))::date + 1))
                    Batches: 21  Memory Usage: 8265kB  Disk Usage: 7400kB
                    Buffers: shared hit=5966, temp read=3332 written=

In [17]:
explain_mv_sql = """
EXPLAIN (ANALYZE, BUFFERS)
WITH target_customer AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t
    ON t.customer_id = mv.customer_id;
"""

with engine.connect() as conn:
    result = conn.execute(text(explain_mv_sql))
    explain_mv = [row[0] for row in result]

print("\n".join(explain_mv))

Nested Loop  (cost=0.83..8.98 rows=1 width=158) (actual time=0.082..0.084 rows=1 loops=1)
  Buffers: shared hit=5 read=3
  ->  Limit  (cost=0.42..0.55 rows=1 width=32) (actual time=0.038..0.039 rows=1 loops=1)
        Buffers: shared hit=4
        ->  Index Only Scan using olist_customers_pkey on olist_customers  (cost=0.42..10897.92 rows=84100 width=32) (actual time=0.037..0.037 rows=1 loops=1)
              Heap Fetches: 1
              Buffers: shared hit=4
  ->  Index Scan using idx_mv_olist_customer_features_customer_id on mv_olist_customer_features mv  (cost=0.42..8.44 rows=1 width=158) (actual time=0.039..0.039 rows=1 loops=1)
        Index Cond: (customer_id = olist_customers.customer_id)
        Buffers: shared hit=1 read=3
Planning:
  Buffers: shared hit=7
Planning Time: 0.590 ms
Execution Time: 0.124 ms


### Reflection

Після виконання `EXPLAIN (ANALYZE, BUFFERS)` я побачила досить велику різницю між звичайним CTE-запитом і Materialized View. Повний CTE-pipeline виконався за 639.932 ms. У плані є `Merge Left Join`, `GroupAggregate`, `Sort` та `Index Scan`. Найбільше роботи тут потрібно для об'єднання таблиць orders і order_items, а потім для групування та підрахунку результатів. Це логічно, тому що запит працює з великою кількістю рядків і кожного разу заново рахує потрібні значення.

Materialized View у цьому випадку набагато зручніша, якщо ці дані потрібно використовувати часто. Запит до MV виконався всього за 0.088 ms. Також у плані видно `Index Scan` по індексу `idx_mv_olist_customer_features_customer_id`. Тобто готові дані можна отримати набагато швидше. Я б оновлювала таку MV раз на день, якщо нові дані не потрібно бачити одразу. Якщо оновлення потрібне тільки іноді, можна робити refresh за потреби.

Звичайний `VIEW` я б використовувала тоді, коли дані часто змінюються і важливо завжди отримувати їх актуальний стан. У такому випадку результат буде рахуватися під час кожного запиту. Це може бути повільніше, але користувач завжди буде бачити актуальні дані.

У recursive CTE я використала дві перевірки для зупинки рекурсії. `path_ids` зберігає вже пройдені customer_id, а `depth < 50` не дозволяє рекурсії йти занадто глибоко. Якщо буде цикл `A → B → A`, повторний перехід до A не відбудеться, тому що A вже буде в `path_ids`.

Для цього pipeline, на мою думку, головний компроміс - це швидкість і актуальність даних. MV працює набагато швидше, але між refresh дані в ній можуть бути трохи застарілими. У цьому випадку я б обрала швидше читання даних, тому що щоденного оновлення для такого сценарію мені було б достатньо.